# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [12]:
from pathlib import Path

# Load the original aviation dataset
aviation_path = Path("data") / "AviationData.csv"
state_codes_path = Path("data") / "USState_Codes.csv"

df_raw = pd.read_csv(
    aviation_path,
    encoding="latin1",
    low_memory=False
)

df_states = pd.read_csv(
    state_codes_path,
    encoding="latin1"
)

print("Aviation data loaded successfully:", df_raw.shape)
print("State codes loaded successfully:", df_states.shape)

Aviation data loaded successfully: (88889, 31)
State codes loaded successfully: (62, 2)


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [13]:
# Work on a copy and standardize text fields without merging distinct makes/models.
df = df_raw.copy()
df.columns = df.columns.astype(str).str.strip()

# Inspect the fields used for the requested filters and outcomes.
important_cols = [
    "Event.Date", "Investigation.Type", "Aircraft.Category", "Amateur.Built",
    "Make", "Model", "Aircraft.Damage", "Total.Fatal.Injuries",
    "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured",
    "Weather.Condition", "Engine.Type", "Number.of.Engines",
    "Broad.phase.of.flight", "Purpose.of.flight"
]
print(df[[c for c in important_cols if c in df.columns]].dtypes)
print("\nMissing values in key columns:")
display(df[[c for c in important_cols if c in df.columns]].isna().sum().sort_values(ascending=False).to_frame("missing_count"))

# Date filtering: retain events from 1983 onward, per the assignment's 40-year rule.
if "Event.Date" not in df.columns:
    raise KeyError("Expected an 'Event.Date' column. Check the source CSV headers.")
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")
df = df.loc[df["Event.Date"].dt.year >= 1983].copy()

# Analyze accidents rather than incidents where the investigation type is available.
if "Investigation.Type" in df.columns:
    df = df.loc[df["Investigation.Type"].astype("string").str.strip().str.lower().eq("accident")].copy()

# The client wants professionally built airplanes. Exclude amateur-built and unknown build status.
if "Amateur.Built" in df.columns:
    built = df["Amateur.Built"].astype("string").str.strip().str.lower()
    df = df.loc[built.eq("no")].copy()

# Keep airplane-category events where that field is present. This excludes helicopters/gliders etc.
if "Aircraft.Category" in df.columns:
    category = df["Aircraft.Category"].astype("string").str.strip().str.lower()
    df = df.loc[category.eq("airplane")].copy()

# Normalize key labels: trim whitespace and standardize case; do not merge different model names.
for col in ["Make", "Model", "Aircraft.Damage", "Weather.Condition", "Engine.Type", "Broad.phase.of.flight", "Purpose.of.flight"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
if "Make" in df.columns:
    df["Make"] = df["Make"].str.upper()
if "Model" in df.columns:
    df["Model"] = df["Model"].str.upper()

print("Rows after date/accident/build/category filters:", len(df))
print("Event date range:", df["Event.Date"].min(), "to", df["Event.Date"].max())
display(df.head())


Event.Date                    str
Investigation.Type            str
Aircraft.Category             str
Amateur.Built                 str
Make                          str
Model                         str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Engine.Type                   str
Number.of.Engines         float64
Broad.phase.of.flight         str
Purpose.of.flight             str
dtype: object

Missing values in key columns:


,missing_count
Aircraft.Category,56602
Broad.phase.of.flight,27165
Total.Serious.Injuries,12510
Total.Minor.Injuries,11933
Total.Fatal.Injuries,11401
Engine.Type,7096
Purpose.of.flight,6192
Number.of.Engines,6084
Total.Uninjured,5912
Weather.Condition,4492


Rows after date/accident/build/category filters: 19937
Event date range: 1983-03-20 00:00:00 to 2022-12-26 00:00:00


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,Skydiving,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016
5960,20001214X44100,Accident,DCA83AA036,1983-08-21,"SILVANA, WA",United States,NaN,NaN,S88,NaN,...,Skydiving,NaN,11.0,2.0,NaN,13.0,VMC,Other,Probable Cause,17-10-2016
6669,20001214X44944,Accident,NYC84LA015,1983-10-28,"MIDDLETOWN, PA",United States,NaN,NaN,NaN,NaN,...,Unknown,NaN,1.0,NaN,NaN,29.0,VMC,Climb,Probable Cause,09-12-2011
6806,20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,NaN,NaN,NaN,1.0,VMC,Climb,Probable Cause,05-05-2011


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [14]:
# Injury metric construction.
# Assumption: blank injury counts are treated as zero (the source commonly leaves zero counts blank).
# Rows with no reported occupants at all receive NaN for the fraction, rather than a false zero-risk score.
injury_cols = ["Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured"]
missing_required = [c for c in injury_cols if c not in df.columns]
if missing_required:
    raise KeyError(f"Missing expected injury columns: {missing_required}")

for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").clip(lower=0)
# Use zero for individual blank categories, while retaining a denominator check below.
df[injury_cols] = df[injury_cols].fillna(0)
df["Total.Occupants"] = df[injury_cols].sum(axis=1)
df["Fatal.Serious.Injuries"] = df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
df["Fatal.Serious.Fraction"] = np.where(
    df["Total.Occupants"] > 0,
    df["Fatal.Serious.Injuries"] / df["Total.Occupants"],
    np.nan
)
df["Any.Fatal.Serious.Injury"] = np.where(
    df["Total.Occupants"] > 0,
    (df["Fatal.Serious.Injuries"] > 0).astype(float),
    np.nan
)
print(df[[*injury_cols, "Total.Occupants", "Fatal.Serious.Injuries", "Fatal.Serious.Fraction"]].describe())


       Total.Fatal.Injuries  Total.Serious.Injuries  Total.Minor.Injuries  \
count          19937.000000             19937.00000          19937.000000   
mean               0.694738                 0.28876              0.218237   
std                6.559137                 2.18752              1.707660   
min                0.000000                 0.00000              0.000000   
25%                0.000000                 0.00000              0.000000   
50%                0.000000                 0.00000              0.000000   
75%                0.000000                 0.00000              0.000000   
max              295.000000               161.00000            200.000000   

       Total.Uninjured  Total.Occupants  Fatal.Serious.Injuries  \
count     19937.000000     19937.000000            19937.000000   
mean          4.903947         6.105683                0.983498   
std          27.268336        28.491958                7.030694   
min           0.000000         0.00000

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [17]:
#Damage outcome: classify known damage labels only

if "Aircraft.damage" not in df.columns:
    raise KeyError("Expected an 'Aircraft.damage' column.")

# Standardize damage descriptions
damage = df["Aircraft.damage"].astype("string").str.strip().str.lower()

# Create destruction indicator
df["Aircraft.Destroyed"] = pd.Series(
    pd.NA, index=df.index, dtype="boolean"
)

# Identify destroyed aircraft
df.loc[
    damage.str.contains("destroy", na=False),
    "Aircraft.Destroyed"
] = True

# Identify aircraft that were not destroyed
known_not_destroyed = (
    damage.notna() &
    ~damage.str.contains("destroy", na=False)
)

df.loc[known_not_destroyed, "Aircraft.Destroyed"] = False

# Display results
print("Damage categories:")
display(df["Aircraft.damage"].value_counts(dropna=False).to_frame("records"))

print("\nAircraft destruction summary:")
display(df["Aircraft.Destroyed"].value_counts(dropna=False).to_frame("records"))

Damage categories:


,records
Aircraft.damage,
Substantial,16965
Destroyed,2316
NaN,434
Minor,147
Unknown,75



Aircraft destruction summary:


,records
Aircraft.Destroyed,
False,17187
True,2316
<NA>,434


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [18]:
# Make cleaning and minimum-sample filter.
# Standardization above trims whitespace and converts makes to uppercase. Missing makes are excluded.
if "Make" not in df.columns:
    raise KeyError("Expected a 'Make' column.")
missing_make_before = int(df["Make"].isna().sum())
df = df.loc[df["Make"].notna() & df["Make"].ne("")].copy()
make_counts = df["Make"].value_counts()
MIN_MAKE_RECORDS = 50
eligible_makes = make_counts[make_counts >= MIN_MAKE_RECORDS].index
df = df.loc[df["Make"].isin(eligible_makes)].copy()
print(f"Rows with missing make removed: {missing_make_before}")
print(f"Makes with at least {MIN_MAKE_RECORDS} records retained: {len(eligible_makes)}")
display(df["Make"].value_counts().head(20).to_frame("accident_records"))

Rows with missing make removed: 2
Makes with at least 50 records retained: 34


,accident_records
Make,
CESSNA,7029
PIPER,3938
BEECH,1392
BOEING,553
MOONEY,358
BELLANCA,217
AIR TRACTOR INC,217
MAULE,215
CIRRUS DESIGN CORP,209


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [19]:
# Exclude rows with no model and build a make+model identifier because model labels can repeat across makes.
if "Model" not in df.columns:
    raise KeyError("Expected a 'Model' column.")
missing_model_before = int(df["Model"].isna().sum())
df = df.loc[df["Model"].notna() & df["Model"].ne("")].copy()
df["Plane.Type"] = df["Make"].astype(str) + " | " + df["Model"].astype(str)
print("Rows with missing model removed:", missing_model_before)
print("Unique models:", df["Model"].nunique())
print("Unique make/model types:", df["Plane.Type"].nunique())
print("Model labels shared by multiple makes (examples):")
model_make_counts = df.groupby("Model")["Make"].nunique().sort_values(ascending=False)
display(model_make_counts[model_make_counts > 1].head(15).to_frame("number_of_makes"))


Rows with missing model removed: 11
Unique models: 1841
Unique make/model types: 1944
Model labels shared by multiple makes (examples):


,number_of_makes
Model,
7GCBC,3
7GCAA,3
7AC,3
8GCBC,3
8KCAB,3
7ECA,3
S2R,3
7EC,3
402B,2


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [20]:
# Exclude # Clean categorical variables used for factor analysis. Keep missing values as 'Unknown' for transparent comparisons.
other_cols = ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]
for col in other_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        print(f"\n{col}: {df[col].nunique(dropna=True)} non-missing categories; missing={df[col].isna().sum()}")
        display(df[col].value_counts(dropna=False).head(20).to_frame("records"))

if "Number.of.Engines" in df.columns:
    df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")



Engine.Type: 7 non-missing categories; missing=2442


,records
Engine.Type,
Reciprocating,12735
<NA>,2442
Turbo Prop,877
Turbo Fan,364
Unknown,67
Turbo Jet,45
Turbo Shaft,8
UNK,1



Weather.Condition: 4 non-missing categories; missing=1504


,records
Weather.Condition,
VMC,13974
<NA>,1504
IMC,871
Unk,147
UNK,43



Number.of.Engines: 5 non-missing categories; missing=1424


,records
Number.of.Engines,
1.0,13130
2.0,1938
<NA>,1424
4.0,28
3.0,15
0.0,4



Purpose.of.flight: 24 non-missing categories; missing=1907


,records
Purpose.of.flight,
Personal,9775
Instructional,2387
<NA>,1907
Aerial Application,721
Business,393
Unknown,253
Positioning,253
Skydiving,156
Aerial Observation,146



Broad.phase.of.flight: 12 non-missing categories; missing=14120


,records
Broad.phase.of.flight,
<NA>,14120
Landing,1105
Takeoff,421
Cruise,233
Approach,209
Maneuvering,126
Taxi,92
Go-around,81
Descent,59


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [21]:
# Clean c# Drop columns that are overwhelmingly missing, while protecting fields required for analysis.
required_for_analysis = {
    "Event.Date", "Make", "Model", "Plane.Type", "Aircraft.Damage", "Aircraft.Destroyed",
    "Total.Fatal.Injuries", "Total.Serious.Injuries", "Total.Minor.Injuries", "Total.Uninjured",
    "Total.Occupants", "Fatal.Serious.Injuries", "Fatal.Serious.Fraction", "Any.Fatal.Serious.Injury",
    "Weather.Condition", "Engine.Type", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"
}
missing_fraction = df.isna().mean()
remove_cols = [c for c in df.columns if missing_fraction[c] > 0.90 and c not in required_for_analysis]
print("Dropping columns with >90% missing values:", remove_cols)
df = df.drop(columns=remove_cols)
print("Final cleaned shape:", df.shape)

Dropping columns with >90% missing values: ['Schedule']
Final cleaned shape: (16539, 36)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [22]:
# Save cleaned data for the separate analysis notebook.
output_path = Path("aviation_accidents_cleaned.csv")
df.to_csv(output_path, index=False)
print(f"Saved {len(df):,} rows and {len(df.columns)} columns to {output_path.resolve()}")


Saved 16,539 rows and 36 columns to /workspaces/dsc-course0-m8-lab/aviation_accidents_cleaned.csv
